# Notebook 1 — Data Preprocessing Fundamentals
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

**Methodology for every topic below:**
1. **Understand the Concept** (Markdown) — explained in my own words.
2. **Demonstrate the Concept** (Markdown) — simple / real-world / business / AI-ML
   example, explaining why the technique is required and what problem it solves.
3. **Implement the Concept** (Python) — with what the code does, why the technique was
   selected, before/after state, why the approach is appropriate, and the ML impact.

**Guiding principle for this sprint:** Identify → Analyze → Clean → Transform →
Validate → Prepare for Machine Learning.

**Dataset:** continuing with **Telco Customer Churn** (7,043 customers, 21 columns) from
Sprint 4 — this notebook grounds every abstract concept in the specific, real
data-quality issues already documented in that sprint's EDA report, rather than
hypothetical examples.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Dataset loaded: 7,043 rows, 21 columns


---
## 1. What is Data Preprocessing?

### Step 1 — Understand the Concept
Data preprocessing is the set of steps that convert raw, as-collected data into a clean,
consistent, and correctly-structured form that a machine learning algorithm can actually
use — everything that happens between "loading a dataset" and "handing it to a model."

### Step 2 — Demonstrate the Concept
**Simple example:** The Telco dataset's `TotalCharges` column arrives as text with 11
blank entries (Sprint 4). Before any model can compute with it, that column needs to be
converted to numbers and those gaps need to be resolved — that conversion and gap-filling
IS data preprocessing.

**AI/ML use case:** No algorithm — not even the most sophisticated deep learning model —
can learn a meaningful pattern from a column stored as the wrong type or containing
disguised missing values; preprocessing is what makes learning possible in the first
place.

### Step 3 — Implement the Concept


In [2]:
print("BEFORE preprocessing:")
print(f"  TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"  Blank entries     : {(df['TotalCharges'].str.strip() == '').sum()}")

df_demo = df.copy()
df_demo['TotalCharges'] = pd.to_numeric(df_demo['TotalCharges'], errors='coerce')

print("\nAFTER preprocessing (type conversion only, gap-filling comes in Notebook 3):")
print(f"  TotalCharges dtype: {df_demo['TotalCharges'].dtype}")
print(f"  NaN entries        : {df_demo['TotalCharges'].isnull().sum()}")


BEFORE preprocessing:
  TotalCharges dtype: str
  Blank entries     : 11

AFTER preprocessing (type conversion only, gap-filling comes in Notebook 3):
  TotalCharges dtype: float64
  NaN entries        : 11


**What this does:** Converts `TotalCharges` from text to numeric, revealing the 11
disguised blanks as proper `NaN` values. **Why this technique:** `pd.to_numeric(...,
errors='coerce')` is the standard way to force a conversion while safely marking
un-convertible values rather than crashing. **Before:** an `object` column with invisible
blanks. **After:** a `float64` column with explicit, visible gaps. **Why appropriate:**
this is the necessary first step before any further cleaning of this column can even be
attempted. **ML impact:** without this single step, `TotalCharges` would be entirely
unusable as a numeric feature — this is the smallest possible example of what
preprocessing means in practice.


---
## 2. Why is Data Preprocessing Required?

### Step 1 — Understand the Concept
Preprocessing is required because real-world data is virtually never collected with
machine learning in mind — it's collected for business operations (billing, record
keeping), and inherits all of that process's quirks, gaps, and inconsistencies. A model
trained on unprocessed data either fails outright or, worse, silently learns the wrong
thing.

### Step 2 — Demonstrate the Concept
**Business example:** If `TotalCharges`'s blanks were fed directly into a model without
handling, most scikit-learn algorithms would simply raise an error and refuse to train at
all — preprocessing isn't optional polish, it's a hard prerequisite.

**AI/ML use case:** Beyond just "the model can't run," poorly preprocessed data (wrong
scales, inconsistent categories, leaked information) can make a model run *successfully*
while performing far worse — or misleadingly better — than a well-preprocessed version of
the same data and algorithm.

### Step 3 — Implement the Concept


In [3]:
from sklearn.linear_model import LogisticRegression

X_raw = df_demo[['tenure', 'MonthlyCharges', 'TotalCharges']]
y = (df['Churn'] == 'Yes').astype(int)

try:
    LogisticRegression().fit(X_raw, y)
    print("Model trained successfully.")
except ValueError as e:
    print(f"Training FAILED with raw (unprocessed) data:\n{type(e).__name__}: {str(e)[:200]}")


Training FAILED with raw (unprocessed) data:
ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and


**What this does:** Attempts to train a simple classifier directly on the
still-has-missing-values `TotalCharges` column. **Why this technique:** deliberately
demonstrating the failure is more convincing than just asserting preprocessing is
"needed." **Before:** raw data with 11 `NaN` values. **After:** (not reached — this cell
demonstrates the failure, not the fix). **Why appropriate:** seeing the actual error
scikit-learn raises makes the requirement concrete rather than abstract. **ML impact:**
this exact failure is why Notebook 3 (Missing Value Handling) exists — every notebook in
this sprint solves a real, demonstrable blocker like this one, not a hypothetical concern.


---
## 3. Raw Data vs Clean Data

### Step 1 — Understand the Concept
Raw data is data exactly as collected or received — with whatever quirks, gaps, or
inconsistencies came with it. Clean data has been through preprocessing: consistent
types, resolved gaps, validated values, and a structure ready for direct use.

### Step 2 — Demonstrate the Concept
**Simple example:** The Telco CSV as downloaded (raw) versus the same dataset after
Sprint 4/5's fixes are applied (clean) — same underlying information, very different
usability.

### Step 3 — Implement the Concept


In [4]:
print("RAW state (as loaded):")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].dtypes)
print(f"Missing values (naive .isnull() count): {df.isnull().sum().sum()}")

print("\nCLEAN state (type-corrected, this notebook's scope):")
print(df_demo[['tenure', 'MonthlyCharges', 'TotalCharges']].dtypes)
print(f"Missing values (now correctly visible): {df_demo.isnull().sum().sum()}")


RAW state (as loaded):
tenure              int64
MonthlyCharges    float64
TotalCharges          str
dtype: object
Missing values (naive .isnull() count): 0

CLEAN state (type-corrected, this notebook's scope):
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object
Missing values (now correctly visible): 11


**What this does:** Directly contrasts the dataset's dtype and missing-value
picture before and after even this notebook's minimal type-correction step.
**Why this technique:** a side-by-side comparison is more concrete than describing "raw"
and "clean" abstractly. **Before:** `TotalCharges` as text, 0 missing values reported
(misleadingly). **After:** `TotalCharges` as float, 11 missing values correctly visible.
**Why appropriate:** this exact comparison format (before/after) is what Notebook 15 of
this sprint will do comprehensively across the whole dataset. **ML impact:** "clean" isn't
a vague quality label — it's a specific, checkable state (correct types, resolved gaps,
validated values) that this whole sprint works toward step by step.


---
## 4. Data Quality

### Step 1 — Understand the Concept
Data quality is an umbrella term covering several distinct dimensions — completeness,
consistency, accuracy, validity, and integrity (each covered as its own topic below) —
that together determine whether a dataset can be trusted for analysis or modeling.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** "Is this dataset good enough to model?" isn't a single yes/no
question — it decomposes into "is it complete? is it consistent? is it accurate? is it
valid? is it internally coherent?" — each of which can pass or fail independently.

### Step 3 — Implement the Concept


In [5]:
quality_snapshot = {
    'Completeness': f"{(1 - df_demo.isnull().sum().sum() / df_demo.size) * 100:.2f}% cells non-missing",
    'Consistency'  : "Checked per-column in Topic 5 below",
    'Accuracy'     : "Checked per-column in Topic 6 below",
    'Validity'     : "Checked per-column in Topic 7 below",
    'Integrity'    : "Checked in Topic 8 below",
}
for dimension, status in quality_snapshot.items():
    print(f"{dimension:<13}: {status}")


Completeness : 99.99% cells non-missing
Consistency  : Checked per-column in Topic 5 below
Accuracy     : Checked per-column in Topic 6 below
Validity     : Checked per-column in Topic 7 below
Integrity    : Checked in Topic 8 below


**What this does:** Frames the five quality dimensions as a checklist applied to
this specific dataset, previewing the next four topics. **Why this technique:** treating
"data quality" as a checklist rather than a single score keeps each dimension separately
actionable. **Before/After:** not applicable — this is a framing topic, not a
transformation. **Why appropriate:** every remaining topic in this section fills in one
row of this checklist with a concrete, re-verified finding. **ML impact:** a dataset can
be 100% complete yet still fail on accuracy or validity — treating quality as
one-dimensional risks missing real problems a completeness check alone wouldn't catch.


---
## 5. Data Consistency

### Step 1 — Understand the Concept
Data consistency means the same underlying value is represented the same way everywhere
it appears — no column has "Male"/"male"/"M" all referring to the same category, and
related columns don't contradict each other.

### Step 2 — Demonstrate the Concept
**Business example:** If `InternetService` said "No" for a customer but `OnlineSecurity`
said "Yes" for that same customer, that would be an internal contradiction — you can't
have online security without internet service.

### Step 3 — Implement the Concept


In [6]:
# Consistency check: no customer should have an internet add-on WITHOUT having internet service
addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
inconsistent_rows = 0
for col in addon_cols:
    bad = df[(df['InternetService'] == 'No') & (df[col] != 'No internet service')]
    inconsistent_rows += len(bad)

print(f"Rows with an internet add-on inconsistent with InternetService='No': {inconsistent_rows}")

# Consistency check: category label spelling/casing
print(f"\nDistinct casing check on 'gender': {df['gender'].unique()}")
print(f"Distinct casing check on 'Contract': {df['Contract'].unique()}")


Rows with an internet add-on inconsistent with InternetService='No': 0

Distinct casing check on 'gender': <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
Distinct casing check on 'Contract': <ArrowStringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str


**What this does:** Cross-checks that every internet add-on column is consistently
"No internet service" for customers with no internet, and checks category spelling
consistency. **Why this technique:** a direct logical cross-check between related columns
catches contradictions a single-column check never would. **Before/After:** both checks
pass with zero problems — this dataset is genuinely consistent. **Why appropriate:** this
mirrors exactly how the shared "No internet service" pattern was first noticed in Sprint
4, Notebook 9, now formalized as an explicit validation. **ML impact:** an inconsistent
dataset would confuse a model with contradictory signals about the same underlying fact;
confirming consistency here means no consistency-repair work is needed for this dataset.


---
## 6. Data Completeness

### Step 1 — Understand the Concept
Data completeness measures how much of the expected data is actually present — missing
values, empty strings, and disguised gaps (like the `TotalCharges` blanks) all reduce
completeness.

### Step 2 — Demonstrate the Concept
**Business example:** The `TotalCharges` column is 99.84% complete after accounting for
the 11 disguised blanks — high, but not perfect, and the specific pattern (all 11 tied to
brand-new customers) matters more than the raw percentage alone.

### Step 3 — Implement the Concept


In [7]:
completeness_pct = (1 - df_demo.isnull().sum() / len(df_demo)) * 100
incomplete_cols = completeness_pct[completeness_pct < 100]
print("Columns that are NOT 100% complete:")
print(incomplete_cols.round(2))


Columns that are NOT 100% complete:
TotalCharges    99.84
dtype: float64


**What this does:** Computes exact completeness percentage per column and isolates
any column below 100%. **Why this technique:** a per-column percentage is more actionable
than a single dataset-wide summary. **Before:** `TotalCharges` appeared 100% complete via
naive `.isnull()`. **After:** correctly shown as 99.84% complete (11 of 7,043 missing).
**Why appropriate:** this reuses Sprint 4's exact finding, now formally categorized under
the "completeness" quality dimension rather than as a standalone anomaly. **ML impact:**
99.84% completeness on one column is a minor, easily-addressed issue (Notebook 3) — very
different from a column at, say, 60% completeness, which would need a much more careful
imputation-vs-drop decision.


---
## 7. Data Accuracy

### Step 1 — Understand the Concept
Data accuracy means the values, where present, correctly reflect reality — a complete,
consistent, valid-looking value can still be simply *wrong* (e.g., a customer's age
recorded as 200), which is what accuracy specifically checks for, distinct from the other
quality dimensions.

### Step 2 — Demonstrate the Concept
**Business example:** A `MonthlyCharges` value of $5,000 would be complete (not missing),
consistent (correctly typed), and even technically valid (a positive number) — but almost
certainly *inaccurate* for this company's actual pricing plans.

### Step 3 — Implement the Concept


In [8]:
print("Accuracy check — plausible range for MonthlyCharges based on known pricing:")
print(f"  Min: {df['MonthlyCharges'].min()}, Max: {df['MonthlyCharges'].max()}")
print(f"  Any value above $200 (implausible for this business)? {(df['MonthlyCharges'] > 200).sum()}")

print("\nAccuracy check — tenure cannot exceed a person's plausible working lifetime:")
print(f"  Any tenure above 100 years (1200 months)? {(df['tenure'] > 1200).sum()}")


Accuracy check — plausible range for MonthlyCharges based on known pricing:
  Min: 18.25, Max: 118.75
  Any value above $200 (implausible for this business)? 0

Accuracy check — tenure cannot exceed a person's plausible working lifetime:
  Any tenure above 100 years (1200 months)? 0


**What this does:** Applies domain-knowledge-based plausibility bounds beyond just
"is it missing" or "is it the right type." **Why this technique:** accuracy checks require
business context, not just statistical rules — a value can be statistically unremarkable
yet still be wrong. **Before/After:** no accuracy violations found — every value falls
within plausible real-world bounds. **Why appropriate:** this connects directly to Sprint
4, Notebook 7's finding of zero univariate outliers — accuracy and outlier-freedom are
related but distinct checks. **ML impact:** an accurate dataset means the model is
learning from values that genuinely reflect the business reality it's meant to predict,
not from silent recording errors.


---
## 8. Data Validity

### Step 1 — Understand the Concept
Data validity means every value conforms to its defined rules — the right format, the
right set of allowed categories, the right range — regardless of whether it's also
*accurate*. A value can be valid (a real category from the accepted set) yet still be
inaccurate in some other sense, and vice versa.

### Step 2 — Demonstrate the Concept
**Business example:** `Churn` must be exactly "Yes" or "No" — any other string (like
"Y", "1", or a blank) would be an invalid value, regardless of whether it might correctly
reflect some real customer status.

### Step 3 — Implement the Concept


In [9]:
valid_categories = {
    'Churn': {'Yes', 'No'},
    'gender': {'Male', 'Female'},
    'Contract': {'Month-to-month', 'One year', 'Two year'},
    'PaymentMethod': {'Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check'},
}

for col, valid_set in valid_categories.items():
    invalid = set(df[col].unique()) - valid_set
    print(f"{col}: invalid values found = {invalid if invalid else 'None'}")


Churn: invalid values found = None
gender: invalid values found = None
Contract: invalid values found = None
PaymentMethod: invalid values found = None


**What this does:** Defines the exact accepted category set for four key columns
and checks for any value outside it. **Why this technique:** explicit allow-lists are the
standard way to validate categorical data, directly foreshadowing Notebook 5's formal
validation rules. **Before/After:** zero invalid values found in any checked column.
**Why appropriate:** this is a direct, code-based formalization of what Sprint 4,
Notebook 9's "checking for unexpected categories" did informally. **ML impact:** invalid
category values, if present, would either break one-hot encoding or silently create a
spurious new category — confirming validity now prevents that failure mode later in this
sprint (Notebook 7).


---
## 9. Data Integrity

### Step 1 — Understand the Concept
Data integrity means the dataset's internal structure and relationships remain intact and
trustworthy — every record has a valid, unique identifier, and any implied relationships
between fields hold true across the whole dataset, not just in a sample.

### Step 2 — Demonstrate the Concept
**Business example:** Every `customerID` should be unique (Sprint 4, Notebook 1) — if two
rows shared the same ID, that would be an integrity violation, since it would be unclear
which record actually describes that customer.

### Step 3 — Implement the Concept


In [10]:
print(f"customerID uniqueness: {df['customerID'].nunique()} unique out of {len(df)} rows")
print(f"Integrity holds: {df['customerID'].nunique() == len(df)}")

# A second integrity check: TotalCharges should be roughly consistent with tenure * MonthlyCharges
# (using the same z-score residual method as Sprint 4, Notebook 7, for a consistent, comparable result)
from scipy import stats as sstats
df_demo['expected_total'] = df_demo['tenure'] * df_demo['MonthlyCharges']
df_demo['residual'] = df_demo['TotalCharges'] - df_demo['expected_total']
large_mismatch = (np.abs(sstats.zscore(df_demo['residual'].fillna(0))) > 3).sum()
print(f"\nRows flagged as a billing mismatch (residual |z| > 3): {large_mismatch}")


customerID uniqueness: 7043 unique out of 7043 rows
Integrity holds: True

Rows flagged as a billing mismatch (residual |z| > 3): 112


**What this does:** Confirms the identifier's uniqueness (structural integrity) and
checks whether the billing fields remain internally coherent (relational integrity).
**Why this technique:** integrity is fundamentally about relationships holding across the
*whole* dataset, so checking `nunique() == len(df)` and a cross-field consistency
tolerance are both appropriate whole-dataset checks. **Before/After:** identifier
integrity holds perfectly; only a small number of rows show a large billing mismatch —
directly the 112 multivariate outliers already investigated and explained in Sprint 4,
Notebook 7. **Why appropriate:** framing that earlier finding under "integrity" here shows
how the same real finding can be classified under different quality dimensions depending
on the lens applied. **ML impact:** the customerID integrity check confirms the identifier
is safe to use for tracking predictions back to individuals; the billing mismatch is
already a documented, accepted anomaly, not a new problem.


---
## 10. Data Leakage

### Step 1 — Understand the Concept
Data leakage happens when information that wouldn't actually be available at prediction
time accidentally makes its way into the training process — causing a model to appear to
perform very well during development, then fail in real, live use, because the leaked
information isn't actually there when a real prediction is needed.

### Step 2 — Demonstrate the Concept
**Business example:** If this dataset had a `cancellation_processed_date` column that's
only ever filled in for customers who churned, using it as a feature would let a model
"predict" churn almost perfectly — not because it learned anything real, but because the
feature itself only exists *after* the outcome already happened.

### Step 3 — Implement the Concept


In [11]:
# Illustrating leakage with a deliberately constructed example column
df_leak_demo = df.copy()
df_leak_demo['had_exit_interview'] = np.where(df_leak_demo['Churn'] == 'Yes', 1, 0)   # only exists AFTER churn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_leaky = df_leak_demo[['tenure', 'had_exit_interview']]
y = (df_leak_demo['Churn'] == 'Yes').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

model = LogisticRegression().fit(X_train, y_train)
print(f"Accuracy WITH the leaked 'had_exit_interview' feature: {model.score(X_test, y_test):.4f}")

X_clean = df_leak_demo[['tenure']]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_clean, y, test_size=0.2, random_state=42)
model2 = LogisticRegression().fit(X_train2, y_train2)
print(f"Accuracy WITHOUT the leaked feature (tenure alone): {model2.score(X_test2, y_test2):.4f}")


Accuracy WITH the leaked 'had_exit_interview' feature: 1.0000
Accuracy WITHOUT the leaked feature (tenure alone): 0.7346


**What this does:** Constructs a deliberately leaky feature (one that's a direct
byproduct of the target itself) and shows how dramatically it inflates model accuracy
compared to a legitimate feature. **Why this technique:** demonstrating leakage's effect
numerically is more convincing than describing it abstractly — the accuracy gap makes the
danger concrete. **Before/After:** near-perfect accuracy with the leaked feature vs. a
much more realistic accuracy without it. **Why appropriate:** this exact pattern (a
feature that could only exist because the outcome already happened) is precisely what
Notebook 13 of this sprint will cover in full depth — this is a preview, not the complete
treatment. **ML impact:** a model "validated" with leaked data will fail in production,
since `had_exit_interview` would never be available for a real, not-yet-churned customer
at prediction time — this is one of the most dangerous, hard-to-notice mistakes in ML
work.


---
## 11. Training Data, 12. Validation Data, 13. Test Data

### Step 1 — Understand the Concept
A dataset used to build a model is typically split into three roles: **training data**
(what the model directly learns patterns from), **validation data** (used to tune
choices like hyperparameters, without letting the model directly learn from it), and
**test data** (used only once, at the very end, to honestly estimate real-world
performance).

### Step 2 — Demonstrate the Concept
**AI/ML use case:** If a churn model's threshold or hyperparameters were tuned by
repeatedly checking performance on the *test* set, the test set would stop being an
honest, unseen estimate of real performance — this is exactly why validation data exists
as a separate, intermediate role.

### Step 3 — Implement the Concept


In [12]:
from sklearn.model_selection import train_test_split

X = df_demo[['tenure', 'MonthlyCharges']]
y = (df_demo['Churn'] == 'Yes').astype(int)

# First split: separate out the test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
# Second split: divide the remainder into train and validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)
# (0.1765 of the remaining 85% works out to ~15% of the original total)

print(f"Training set  : {len(X_train):,} rows ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(X_val):,} rows ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test set      : {len(X_test):,} rows ({len(X_test)/len(df)*100:.1f}%)")


Training set  : 4,929 rows (70.0%)
Validation set: 1,057 rows (15.0%)
Test set      : 1,057 rows (15.0%)


**What this does:** Splits the dataset into three roles using a standard
~70/15/15 ratio, with `stratify=y` preserving the churn class ratio in every split
(directly required by Sprint 4, Notebook 10's finding of moderate class imbalance).
**Why this technique:** a two-step `train_test_split` call is the standard scikit-learn
pattern for a three-way split, since the function itself only splits into two parts at a
time. **Before:** one full dataset. **After:** three separate, non-overlapping subsets.
**Why appropriate:** `stratify=y` is specifically appropriate here because of the
documented 73.5%/26.5% imbalance. **ML impact:** this three-way split is what makes
honest model evaluation possible — full mechanics and the leakage risks of getting this
wrong are covered in depth in Notebook 12.


---
## 14. Preprocessing Pipeline — The Complete Workflow

### Step 1 — Understand the Concept
A preprocessing pipeline is the ordered sequence of every cleaning and transformation
step applied to a dataset, structured so the exact same steps can be applied
consistently — critically, fitted only on training data and then applied identically to
validation/test data, never the reverse.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Scikit-learn's `Pipeline` and `ColumnTransformer` (covered in full in
Notebook 14 of this sprint) exist specifically to enforce this "fit on train, apply
everywhere" discipline automatically, rather than relying on an engineer to remember it
manually for every step.

### Step 3 — Implement the Concept


In [13]:
workflow_steps = [
    ("1. Identify",  "Load the dataset; inspect types, missing values, duplicates (this notebook + Notebook 2)"),
    ("2. Analyze",   "Understand WHY each problem exists (MCAR/MAR/MNAR, error vs. genuine value)"),
    ("3. Clean",     "Handle missing values (Nb 3), duplicates (Nb 4), validate (Nb 5), treat outliers (Nb 6)"),
    ("4. Transform", "Encode categoricals (Nb 7), scale numerics (Nb 8), transform skewed features (Nb 9)"),
    ("5. Select",    "Feature selection (Nb 10), handle class imbalance (Nb 11)"),
    ("6. Split",     "Train/validation/test split, with correct fit-on-train discipline (Nb 12, 13)"),
    ("7. Validate",  "Confirm the final dataset is genuinely ML-ready (Nb 15, 16)"),
]
for step, description in workflow_steps:
    print(f"{step:<14}: {description}")


1. Identify   : Load the dataset; inspect types, missing values, duplicates (this notebook + Notebook 2)
2. Analyze    : Understand WHY each problem exists (MCAR/MAR/MNAR, error vs. genuine value)
3. Clean      : Handle missing values (Nb 3), duplicates (Nb 4), validate (Nb 5), treat outliers (Nb 6)
4. Transform  : Encode categoricals (Nb 7), scale numerics (Nb 8), transform skewed features (Nb 9)
5. Select     : Feature selection (Nb 10), handle class imbalance (Nb 11)
6. Split      : Train/validation/test split, with correct fit-on-train discipline (Nb 12, 13)
7. Validate   : Confirm the final dataset is genuinely ML-ready (Nb 15, 16)


**What this does:** Lays out this entire sprint's remaining notebooks as one
ordered, end-to-end workflow. **Why this technique:** presenting the roadmap explicitly
here, in the very first notebook, gives every later notebook a clear place in the overall
process rather than feeling like an isolated topic. **Before/After:** not applicable —
this is a planning topic. **Why appropriate:** matches this sprint's own stated guiding
principle (Identify → Analyze → Clean → Transform → Validate → Prepare for ML)
exactly. **ML impact:** following a consistent, ordered pipeline (rather than ad-hoc
fixes applied in a random order) is what makes preprocessing reproducible — essential
both for this sprint's own documentation requirements and for any real production ML
system.


---
## Summary

| Concept | Finding for THIS dataset |
|---|---|
| What is Preprocessing? | Converting raw TotalCharges (text, disguised blanks) into usable numeric data |
| Why Required? | A model literally fails to train on the raw, unconverted column |
| Raw vs Clean | Same data, dramatically different usability before/after type correction |
| Data Quality (umbrella) | Decomposes into completeness, consistency, accuracy, validity, integrity |
| Consistency | No contradictions found between InternetService and its add-on columns |
| Completeness | 99.84% complete overall; TotalCharges' 11 gaps are the only exception |
| Accuracy | No implausible values found in MonthlyCharges or tenure |
| Validity | No invalid category labels found in any spot-checked column |
| Integrity | customerID is a valid unique identifier; 112 rows show an accepted, explained billing anomaly |
| Data Leakage | Demonstrated numerically: a leaked feature inflates accuracy misleadingly |
| Train/Val/Test | Stratified 70/15/15 split, preserving the churn class ratio |
| Preprocessing Pipeline | This sprint's full remaining roadmap, notebook by notebook |

**Next notebook:** `02_Data_Type_Handling.ipynb` — identifying and correctly converting
every column's data type, including the `TotalCharges` fix applied only partially in this
notebook.
